In [2]:
# SoRL in modded-gpt compatible fashion (for ultra fast pre-training)
# 1. pre-training demands simple model architecture, even .generate function can be wrapped around the trained model afterwards
# 2. no need to include 'kv-cache' for the pre-training experiment here

In [ ]:
from sorl.model import CausalSelfAttention, Block, GPTConfig
import torch 

# mock input 
x = torch.randn(2, 1024, 768)

config = GPTConfig()
attn = CausalSelfAttention(dim=768, n_head=6)
block = Block(config=config)

y, v1 = attn(x)
x, v1 = block(x, v1, x, None)

In [3]:
import torch 
from sorl.gat import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)


token_ids = torch.randint(0, 128 + 8, (2, 4))
idx = token_ids[:, :-1].contiguous()
target = token_ids[:, 1:].contiguous()


# forward pass ()
ppt = model(idx, target, 1024)

In [4]:
from sorl.gat import parallel_denoise
from sorl.gat import generate 


num_iterations = 5 
memory_span = 1024 
temperature = 0.0

parallel_denoise(model, idx, num_iterations=5, memory_span=1024, temperature=0.0)

generate(model, idx, max_new_tokens=5, abstraction_interval=3)


tensor([[ 42,  86, 110, 129,   0,   0, 129,   0],
        [ 46,  21,   4, 129,   0,   0, 129,   0]])

In [24]:
idx = torch.tensor([
    [4, 129, 129, 129],
    [3, 10, 129, 129]
])

In [5]:
import torch 
from sorl.gat_act import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)

from sorl.gat_act import infer_level

# idx = torch.randint(0, model.vocab_sizes.sum(), (2, 4)).contiguous()
idx = torch.tensor([
    [4, 129, 130, 129],
    [3, 10, 130, 129]
]).contiguous()
levels = infer_level(idx, model.vocab_sizes)
abstract_mask = (levels > 0)

In [6]:
from sorl.gat_act import search, generate, recursion

refined_idx, logits = search(model, idx, max_iterations=1, n_continuous=0, memory_span=1024, temperature=0.0, K=3)

# extended_idx = generate(model, idx, max_new_tokens=10, K=3)

# recursion(model, idx, max_iterations=1, n_continuous=0, memory_span=1024, temperature=0.0)


In [1]:
# --- Compatibility with SoRL algorithm --- 

# (1). search == parallel_denoise
#      this function is directly used to sample abstract tokens
#      we need a rythmic padding function to work on a specific seq
#      but we don't need to worry about context limit etc. aka data 
#      loader does not need to be changed

# Thought 1. 
# - since the model can process batch data, we should re-use functions
#   developed in 'sorl.py' this avoids the extra 'flattened duplicate & selection' gadget

import torch
from sorl.gat_act import GAT, GATConfig, get_level_mask_tokens
from sorl.sorl import SORLConfig, heuristic_rollout
import torch.nn.functional as F

# 1. Create model config and instantiate GAT
gat_config = GATConfig(
    vocab_sizes=[50304, 8],  # Level 0: 128 tokens, Level 1: 8 abstract tokens
    n_layer=12,
    n_head=6,
    n_embd=768,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

model = GAT(gat_config)

# 2. Add missing attributes needed by sorl.py
model.level_mask_tokens = get_level_mask_tokens(model.vocab_sizes)  # [0, 128]
model.full_vocab_size_list = gat_config.vocab_sizes

# 3. Create SORLConfig
sorl_config = SORLConfig(
    n=4,                    # Number of candidate rollouts
    temperature=1.0,        # Sampling temperature
    K=5,                    # Rhythmic stride (abstract token every K positions)
    l=1,                    # Level to search (level 1 = abstract)
    steps=3,                # Steps for chunk-wise denoise
    max_t_search=10,        # Max timestamps to search
    use_rhythmic_placeholders=True,
    use_spike_placeholders=False,
)

# 4. Create dummy data
batch_size = 2
seq_len = 50
data = torch.randint(0, 128, (batch_size, seq_len), device=model.device)

print("Model vocab_sizes:", model.vocab_sizes)
print("Level mask tokens:", model.level_mask_tokens)
print("Data shape:", data.shape)


Model vocab_sizes: tensor([50304,     8])
Level mask tokens: tensor([    0, 50304, 50312])
Data shape: torch.Size([2, 50])


In [2]:
# heuristic rollout
import os
from pathlib import Path

# load data shard
# ---------------
file = Path("data/fineweb10B/fineweb_train_000002.bin")
header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
assert header[0] == 20240520, "magic number mismatch in the data .bin file"
assert header[1] == 1, "unsupported version"
num_tokens = int(header[2]) # number of tokens (claimed)
with file.open("rb", buffering=0) as f:
    tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # avoid pin_memory copy by @YouJiacheng
    f.seek(256 * 4)
    nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
    assert nbytes == 2 * num_tokens, "number of tokens read does not match header"

idx = tokens[:800].unsqueeze(0)
# ----------------- 

In [3]:
from sorl.neo_utils import sorl_rollout
from sorl.gat_act import BOS_TOKEN_ID
import torch 

# Example: rollout with 1 greedy + 2 stochastic samples
tokens = torch.tensor([[BOS_TOKEN_ID, 2, 3, 4, 4, 3, 2, 1, BOS_TOKEN_ID, 4, 5, 6, 1]])
search_data = sorl_rollout(tokens, model, n=3, K=3, max_iterations=1, 
                           n_continuous=0, memory_span=1024, temperature=1.0)

print(f"Search data shape: {search_data.shape}")
print(f"First rollout (greedy): {search_data[0]}")
print(f"Second rollout (stochastic): {search_data[1]}")
print(f"Third rollout (stochastic): {search_data[2]}")



Search data shape: torch.Size([3, 16])
First rollout (greedy): tensor([50256,     2,     3,     4, 50305,     4,     3,     2, 50305,     1,
        50256,     4,     5,     6, 50305,     1])
Second rollout (stochastic): tensor([50256,     2,     3,     4, 50310,     4,     3,     2, 50310,     1,
        50256,     4,     5,     6, 50307,     1])
Third rollout (stochastic): tensor([50256,     2,     3,     4, 50311,     4,     3,     2, 50308,     1,
        50256,     4,     5,     6, 50309,     1])


In [5]:
from sorl.gat_act import recursion, BOS_TOKEN_ID

_, ppt = recursion(model, search_data, max_iterations=1, n_continuous=0, do_discrete=False)
ppt = ppt.reshape(search_data.shape[0], -1)

# Reflection 1. 
# - ACT did not consider masking out first token's argmax, so threshold based stopping is more meaningful



# --- avg ppt for each sample in each rollout --- 

# --- get argmax rollout for each sample ---

In [ ]:
ppt.shape, search_data.shape


ppt_idx = (search_data == BOS_TOKEN_ID).cumsum(dim=1)[:, 1:]

# --- avg ppt for each sample (for each sequence) --- 



# --- get argmax rollout for each sample (across rollouts) --- 

# --- stitch per-sample argmax rollout into one sequence --- 


# SoRL Rollout Pipeline

The complete SoRL rollout pipeline consists of:

1. **`sorl_rollout`**: Generate n rollouts (1 greedy + n-1 stochastic)
   - Inserts rhythmic placeholder tokens at stride K
   - Fills placeholders using search with different temperatures
   - Returns all n rollouts

2. **`compute_perplexity_per_document`**: Evaluate quality of each rollout
   - Computes perplexity per document for each rollout
   - Lower perplexity = better prediction quality
   - Returns matrix of shape (n_rollouts, n_documents)

3. **`select_best_rollouts`**: Stitch best predictions together
   - Selects best rollout per document (lowest perplexity)
   - Stitches selected segments into final sequence
   - Returns single optimized sequence

This implements the SoRL algorithm where multiple rollouts are sampled and the best segments are selected based on perplexity.
